# 06. Main policy optimization

This notebook solves the frozen three-policy promotion problem under the calibrated empirical-Bayes demand-displacement model.

The main economic specification allows predicted baseline demand to vary across weeks while holding regular prices and unit costs fixed at product-level values. Realized weekly cost variation is excluded because inferred costs are strongly associated with the historical promotion calendar and cannot be treated as action-independent inputs under arbitrary counterfactual schedules.

The policy comparison is sequential: a myopic calendar, πN (forward-looking with full inherited displacement but no candidate-promotion state additions), and a displacement-aware dynamic calendar are all evaluated under the same full model. The main specification explicitly assumes a fresh start: I₁=0 for every product and draw, with every product initially cooldown-eligible.

## 1. Imports, paths, and pre-specified configuration

In [1]:
from __future__ import annotations

from pathlib import Path
import hashlib
import subprocess
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Suppress pandas transition warnings so notebook output focuses on diagnostics.
warnings.filterwarnings("ignore", category=FutureWarning)

# Locate the repository root from either the project root or notebooks directory.
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        root
        for root in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (root / "pyproject.toml").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Run the notebook from the repository or notebooks directory."
    )

# Make the local package importable without requiring an editable installation.
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import dynamic_promotion_planning.policy as policy

from dynamic_promotion_planning.policy import (
    PlanningSpec,
    build_weekly_economic_profiles,
    coerce_action_sets,
    demand_multiplier_audit,
    maximum_feasible_promotions,
    prepare_behavioral_draws,
    prepare_support_table,
    run_policy_grid,
    run_three_policy_grid,
    save_pickle,
)
from dynamic_promotion_planning.config import load_analysis_config
from dynamic_promotion_planning.forecast_audit import audit_forecast_information
from dynamic_promotion_planning.policy_workflow import (
    demand_only_profiles,
    load_or_build_schedule_system,
)

# Canonical repository directories used throughout the notebook.
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DEMAND_ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "demand"
CALIBRATION_ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "calibration"

In [2]:
# Define policy artifacts, cache files, and final reporting directories.
# Preserve the prior EB policy run; this price-consistent rerun writes separately.
EB_VERSION = "empirical_bayes_price_consistent"
POLICY_ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "policy" / EB_VERSION
CACHE_DIR = PROJECT_ROOT / "artifacts" / "cache" / "policy" / EB_VERSION
TABLE_DIR = PROJECT_ROOT / "results" / EB_VERSION / "tables"
FIGURE_DIR = PROJECT_ROOT / "results" / EB_VERSION / "figures"

# Create output directories before any optimization artifacts are written.
for directory in [
    PROCESSED_DIR,
    DEMAND_ARTIFACT_DIR,
    CALIBRATION_ARTIFACT_DIR,
    POLICY_ARTIFACT_DIR,
    CACHE_DIR,
    TABLE_DIR,
    FIGURE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Analysis module:", Path(policy.__file__).resolve())

Project root: C:\Users\janza\Desktop\dynamic_promotion_planning_renamed_workflow_0.3.0
Analysis module: C:\Users\janza\Desktop\dynamic_promotion_planning_renamed_workflow_0.3.0\src\dynamic_promotion_planning\policy.py


In [3]:
# Load the pre-specified policy and washout-selection design.
analysis_config = load_analysis_config()
policy_config = analysis_config.policy
washout_config = analysis_config.washout_selection

DECISION_HORIZON = policy_config.decision_horizon
WASHOUT_HORIZONS = list(policy_config.washout_horizons)
COOLDOWN = policy_config.cooldown_weeks
MAX_PROMOTIONS = maximum_feasible_promotions(DECISION_HORIZON, COOLDOWN)
DISCOUNT_FACTOR = policy_config.discount_factor
CAPACITIES = list(policy_config.weekly_capacities)
ECONOMIC_PROFILE_MODE = policy_config.economic_profile

if not WASHOUT_HORIZONS:
    raise ValueError("At least one washout horizon must be configured.")

if not CAPACITIES:
    raise ValueError("At least one weekly capacity must be configured.")

grid_start = float(policy_config.reimbursement_grid_start)
grid_stop = float(policy_config.reimbursement_grid_stop)
grid_step = float(policy_config.reimbursement_grid_step)

if grid_step <= 0:
    raise ValueError("The contract-generosity grid step must be positive.")

if grid_start > grid_stop:
    raise ValueError("The contract-generosity grid bounds are reversed.")

regular_grid = np.arange(
    grid_start,
    grid_stop + 0.5 * grid_step,
    grid_step,
)

ALPHA_GRID = np.round(regular_grid, 8)

if ALPHA_GRID.size == 0:
    raise ValueError("The configured contract-generosity grid is empty.")

MAXIMUM_WASHOUT = max(WASHOUT_HORIZONS)

# Computational controls not currently stored in PolicyConfig.
START_TEST_WEEK_OFFSET = 0
COMPUTE_SECOND_BEST_ON_FULL_GRID = True
MILP_TIME_LIMIT_SECONDS = None

# Retain the empirical-support design in the output artifact for provenance.
BASELINE_SUPPORT = analysis_config.support

VDO_STABILITY_TOLERANCE = washout_config.vdo_stability_tolerance
TERMINAL_STATE_TOLERANCE = washout_config.terminal_state_tolerance
SCHEDULE_BATCH_SIZE = 256

print(
    "Policy grid:",
    f"{len(ALPHA_GRID)} alpha values, "
    f"{len(CAPACITIES)} capacities, "
    "full reimbursement-share range.",
)

Policy grid: 101 alpha values, 4 capacities, full reimbursement-share range.


## 2. Load calibrated behavior, supported actions, and held-out demand profiles

In [4]:
# Define all upstream artifacts required for policy optimization.
EB_CALIBRATION_DIR = CALIBRATION_ARTIFACT_DIR / "empirical_bayes"
PRODUCT_DRAW_PATH = EB_CALIBRATION_DIR / "product_behavioral_draws.pkl"
# Notebook 05's EB calibration/support outputs are immutable inputs.
# Only policy outputs below use the price-consistent versioned directory.
PRODUCT_ACTION_SUPPORT_PATH = (
    PROJECT_ROOT / "results" / "empirical_bayes" / "tables"
    / "supported_action_clusters.csv"
)
PRODUCT_ACTION_SET_PATH = EB_CALIBRATION_DIR / "supported_actions.pkl"
SELECTED_SAMPLE_PATH = PROCESSED_DIR / "paper_selected_sample.parquet"
DEMAND_PREDICTION_PATH = DEMAND_ARTIFACT_DIR / "policy_common_origin_predictions.pkl"

required_paths = [
    PRODUCT_DRAW_PATH,
    PRODUCT_ACTION_SUPPORT_PATH,
    PRODUCT_ACTION_SET_PATH,
    SELECTED_SAMPLE_PATH,
    DEMAND_PREDICTION_PATH,
]

# Fail before optimization if any preceding notebook has not produced its output.
missing_paths = [path for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(
        "Required upstream artifacts are missing:\n"
        + "\n".join(f"  - {path}" for path in missing_paths)
    )

# Load and normalize the calibrated behavioral-draw representation.
raw_draws = pd.read_pickle(PRODUCT_DRAW_PATH)
if raw_draws.empty:
    raise RuntimeError("The product behavioral-draw artifact is empty.")

draw_frame, draws_by_product = prepare_behavioral_draws(raw_draws)

# Load the empirical support table used to diagnose selected actions.
raw_support_table = pd.read_csv(PRODUCT_ACTION_SUPPORT_PATH)
if raw_support_table.empty:
    raise RuntimeError("The empirical action-support table is empty.")

In [5]:
# Normalize the support table and recover the exact product action grids.
support_table = prepare_support_table(raw_support_table)
raw_action_artifact = pd.read_pickle(PRODUCT_ACTION_SET_PATH)

products = sorted(draws_by_product)
if not products:
    raise RuntimeError("No calibrated products are available for optimization.")

action_sets = coerce_action_sets(raw_action_artifact, products)

# Load held-out data used to construct counterfactual weekly profiles.
selected_sample = pd.read_parquet(SELECTED_SAMPLE_PATH)
demand_predictions = pd.read_pickle(DEMAND_PREDICTION_PATH)

if selected_sample.empty:
    raise RuntimeError("The selected empirical sample is empty.")

if demand_predictions.empty:
    raise RuntimeError("The demand-prediction artifact is empty.")

# Report only a compact input summary in this computational notebook.
input_summary = pd.DataFrame(
    {
        "products": [len(products)],
        "behavioral_draws": [len(draw_frame)],
        "supported_positive_actions": [
            sum(len(values) - 1 for values in action_sets.values())
        ],
        "selected_sample_rows": [len(selected_sample)],
        "prediction_rows": [len(demand_predictions)],
    }
)

display(input_summary)

,products,behavioral_draws,supported_positive_actions,selected_sample_rows,prediction_rows
0,8,20780,24,230348,47808


In [6]:
# Build held-out weekly profiles once for the longest evaluation horizon.
# The main specification retains predicted demand variation while fixing price
# and cost factors to one, thereby avoiding action-dependent historical costs.
weekly_profile_table_raw, weekly_profiles_raw, source_weeks = (
    build_weekly_economic_profiles(
        selected_sample=selected_sample,
        demand_predictions=demand_predictions,
        products=products,
        decision_horizon=DECISION_HORIZON,
        maximum_washout=MAXIMUM_WASHOUT,
        start_test_week_offset=START_TEST_WEEK_OFFSET,
        model_name="product_promotion",
    )
)

if not source_weeks:
    raise RuntimeError("No source weeks were selected for the policy horizon.")

weekly_profiles = demand_only_profiles(weekly_profiles_raw)
weekly_profile_table = weekly_profile_table_raw.copy()

# Preserve the original price and cost factors for auditability before replacing
# them with the fixed-profile values used by the policy comparison.
weekly_profile_table["source_price_factor"] = weekly_profile_table["price_factor"]
weekly_profile_table["source_cost_factor"] = weekly_profile_table["cost_factor"]
weekly_profile_table["price_factor"] = 1.0
weekly_profile_table["cost_factor"] = 1.0
weekly_profile_table["economic_profile_mode"] = ECONOMIC_PROFILE_MODE

# Confirm that the transformed profile passed to optimization is demand-only.
assert all(
    np.allclose(values["price_factor"], 1.0)
    for values in weekly_profiles.values()
)
assert all(
    np.allclose(values["cost_factor"], 1.0)
    for values in weekly_profiles.values()
)

In [7]:
# Restrict the forecast-information audit to the held-out predictions from the
# demand model used by the optimizer.
audit_prediction_rows = demand_predictions.copy()

if "split" in audit_prediction_rows.columns:
    audit_prediction_rows = audit_prediction_rows.loc[
        audit_prediction_rows["split"].astype(str).str.lower().eq("test")
    ]

if "model" in audit_prediction_rows.columns:
    audit_prediction_rows = audit_prediction_rows.loc[
        audit_prediction_rows["model"].astype(str).eq("product_promotion_depth")
    ]

if audit_prediction_rows.empty:
    raise RuntimeError(
        "No test predictions from the product-promotion model remain for audit."
    )

# Classify whether the weekly profiles constitute a genuinely ex-ante forecast
# or a conditional counterfactual based on information revealed later.
forecast_information_audit = audit_forecast_information(
    predictions=audit_prediction_rows,
    selected_weeks=source_weeks,
    planning_origin_week=min(source_weeks),
    future_covariates_verified=bool(
        {"prediction_design", "origin_predictors_verified"}.issubset(
            audit_prediction_rows.columns
        )
        and audit_prediction_rows["prediction_design"].astype(str)
        .eq("ex_ante_fixed_grid").all()
        and audit_prediction_rows["origin_predictors_verified"].astype(bool).all()
    ),
)

display(pd.DataFrame([forecast_information_audit.to_dict()]))

if forecast_information_audit.classification != policy_config.forecast_information_mode:
    raise AssertionError(
        "Configured policy information mode does not match the audited forecast path."
    )

if forecast_information_audit.classification != "ex_ante_common_origin":
    print(
        "Forecast-information classification:",
        forecast_information_audit.classification,
    )

,planning_origin_week,first_profile_week,last_profile_week,profile_week_count,row_count,fit_metadata_available,maximum_fit_week,within_horizon_refit_detected,common_origin_fit_verified,future_covariates_verified,classification
0,326,326,375,48,31872,True,325,False,True,True,ex_ante_common_origin


In [8]:
# Summarize and validate the economic profiles without displaying a long
# product-level audit table.
profile_audit = (
    weekly_profile_table.groupby("upc", observed=True)
    .agg(
        demand_factor_min=("demand_factor", "min"),
        demand_factor_max=("demand_factor", "max"),
        demand_factor_std=("demand_factor", "std"),
        price_factor_min=("price_factor", "min"),
        price_factor_max=("price_factor", "max"),
        cost_factor_min=("cost_factor", "min"),
        cost_factor_max=("cost_factor", "max"),
    )
    .reset_index()
)

for column in [
    "price_factor_min",
    "price_factor_max",
    "cost_factor_min",
    "cost_factor_max",
]:
    if not np.allclose(profile_audit[column], 1.0):
        raise AssertionError(
            f"Economic-profile audit failed for {column}."
        )

print(
    "Economic-profile audit passed:",
    f"{len(source_weeks)} source weeks and "
    f"{profile_audit['upc'].nunique()} products.",
)

Economic-profile audit passed: 48 source weeks and 8 products.


## 3. Terminal-tail evaluation diagnostic

Candidate washout horizons are evaluated here. The substantive washout table and figure are presented in Notebook 07.

In [9]:
# Evaluate each candidate washout horizon over the complete reimbursement grid.
washout_runs = []
washout_systems = {}

for washout_horizon in WASHOUT_HORIZONS:
    planning = PlanningSpec(
        decision_horizon=DECISION_HORIZON,
        washout_horizon=washout_horizon,
        cooldown=COOLDOWN,
        max_promotions=MAX_PROMOTIONS,
        discount_factor=DISCOUNT_FACTOR,
        reimbursement_min=float(ALPHA_GRID.min()),
        reimbursement_max=float(ALPHA_GRID.max()),
    )

    cache_path = CACHE_DIR / f"washout_check_w{washout_horizon}.pkl"

    schedule_system = load_or_build_schedule_system(
        cache_path=cache_path,
        planning=planning,
        alpha_grid=ALPHA_GRID,
        draws_by_product=draws_by_product,
        weekly_profiles=weekly_profiles,
        action_sets=action_sets,
        batch_size=SCHEDULE_BATCH_SIZE,
    )
    washout_systems[washout_horizon] = schedule_system

    run = run_policy_grid(
        schedule_system=schedule_system,
        draws_by_product=draws_by_product,
        weekly_profiles=weekly_profiles,
        action_sets=action_sets,
        support_table=support_table,
        alpha_values=ALPHA_GRID,
        capacities=CAPACITIES,
        compute_second_best=False,
        time_limit_seconds=MILP_TIME_LIMIT_SECONDS,
    )

    run["results"]["economic_profile_mode"] = ECONOMIC_PROFILE_MODE
    washout_runs.append(run)

if not washout_runs:
    raise RuntimeError("No washout policy runs were completed.")

Loaded current cache: washout_check_w36.pkl


In [10]:
# Combine washout results and calculate the change in VDO relative to the
# preceding candidate horizon within each reimbursement share and capacity.
washout_results = (
    pd.concat(
        [run["results"] for run in washout_runs],
        ignore_index=True,
    )
    .sort_values(["capacity", "alpha", "washout_horizon"])
    .reset_index(drop=True)
)

if washout_results.empty:
    raise RuntimeError("Washout evaluation produced no policy results.")

washout_results["vdo_change_from_previous"] = (
    washout_results.groupby(["capacity", "alpha"], observed=True)["vdo"].diff()
)

stability_rows = []

In [11]:
# Apply the pre-specified stabilization criteria to every positive washout.
for washout_horizon in [
    value for value in WASHOUT_HORIZONS if value > 0
]:
    current = washout_results.loc[
        washout_results["washout_horizon"].eq(washout_horizon)
    ]

    if current.empty:
        raise RuntimeError(
            f"No policy results were returned for washout {washout_horizon}."
        )

    maximum_vdo_change = float(
        current["vdo_change_from_previous"].abs().max()
    )
    maximum_terminal_state = float(
        current[
            [
                "maximum_terminal_state_dynamic",
                "maximum_terminal_state_myopic",
            ]
        ].max().max()
    )

    stability_rows.append(
        {
            "washout_horizon": washout_horizon,
            "maximum_absolute_vdo_change": maximum_vdo_change,
            "maximum_terminal_state": maximum_terminal_state,
            "vdo_stable": (
                maximum_vdo_change <= VDO_STABILITY_TOLERANCE
            ),
            "terminal_state_small": (
                maximum_terminal_state <= TERMINAL_STATE_TOLERANCE
            ),
        }
    )

washout_stability = pd.DataFrame(stability_rows)

if washout_stability.empty:
    raise RuntimeError(
        "Washout stability cannot be assessed without a positive horizon."
    )

eligible_stable = washout_stability.loc[
    washout_stability["vdo_stable"]
    & washout_stability["terminal_state_small"]
]

In [12]:
# Select the shortest stable washout; fall back to the longest configured
# horizon when no candidate satisfies both criteria.
if eligible_stable.empty:
    SELECTED_WASHOUT = MAXIMUM_WASHOUT
    selection_reason = "fallback to longest configured horizon"
else:
    SELECTED_WASHOUT = int(
        eligible_stable["washout_horizon"].min()
    )
    selection_reason = "first horizon satisfying both criteria"

print(
    "Selected washout:",
    SELECTED_WASHOUT,
    f"({selection_reason})",
)

Selected washout: 36 (fallback to longest configured horizon)


## 4. Main policy computation

The complete grid is solved and validated here. Substantive policy tables and figures are produced in Notebook 07 from the saved artifact.

In [13]:
# Build the schedule system for the complete funding-capacity grid using the
# washout selected above.
primary_planning = PlanningSpec(
    decision_horizon=DECISION_HORIZON,
    washout_horizon=SELECTED_WASHOUT,
    cooldown=COOLDOWN,
    max_promotions=MAX_PROMOTIONS,
    discount_factor=DISCOUNT_FACTOR,
    reimbursement_min=float(ALPHA_GRID.min()),
    reimbursement_max=float(ALPHA_GRID.max()),
)

# Key the cache to the exact alpha grid so a changed interior grid point cannot
# silently reuse a schedule system built under an earlier specification.
alpha_grid_key = hashlib.sha256(
    np.asarray(ALPHA_GRID, dtype=np.float64).tobytes()
).hexdigest()[:12]

primary_cache_path = (
    CACHE_DIR
    / f"full_grid_w{SELECTED_WASHOUT}_{alpha_grid_key}.pkl"
)

primary_schedule_system = load_or_build_schedule_system(
    cache_path=primary_cache_path,
    planning=primary_planning,
    alpha_grid=ALPHA_GRID,
    draws_by_product=draws_by_product,
    weekly_profiles=weekly_profiles,
    action_sets=action_sets,
    batch_size=SCHEDULE_BATCH_SIZE,
)

# πN retains the common fresh-start state but ignores new-promotion displacement.
naive_draws_by_product = draws_by_product
naive_cache_path = CACHE_DIR / f"naive_grid_w{SELECTED_WASHOUT}_{alpha_grid_key}.pkl"
naive_schedule_system = load_or_build_schedule_system(
    cache_path=naive_cache_path,
    planning=primary_planning,
    alpha_grid=ALPHA_GRID,
    draws_by_product=naive_draws_by_product,
    weekly_profiles=weekly_profiles,
    action_sets=action_sets,
    batch_size=SCHEDULE_BATCH_SIZE,
    add_new_promotion_displacement=False,
)

# Solve dynamic and myopic policies under identical inputs at every grid point.
primary_run = run_policy_grid(
    schedule_system=primary_schedule_system,
    draws_by_product=draws_by_product,
    weekly_profiles=weekly_profiles,
    action_sets=action_sets,
    support_table=support_table,
    alpha_values=ALPHA_GRID,
    capacities=CAPACITIES,
    compute_second_best=COMPUTE_SECOND_BEST_ON_FULL_GRID,
    time_limit_seconds=MILP_TIME_LIMIT_SECONDS,
)

policy_results = (
    primary_run["results"]
    .copy()
    .sort_values(["capacity", "alpha"])
    .reset_index(drop=True)
)

if policy_results.empty:
    raise RuntimeError("The primary policy grid produced no results.")

policy_results["economic_profile_mode"] = ECONOMIC_PROFILE_MODE

Loaded current cache: full_grid_w36_6ff5a490c4af.pkl
Loaded current cache: naive_grid_w36_6ff5a490c4af.pkl


### 4.1 Computational validity checks

The dynamic feasible set contains the calendar generated by the myopic policy. Dynamic value must therefore weakly exceed myopic value when both are evaluated under the same inputs and objective.

Dynamic value must also be weakly nondecreasing in weekly capacity. Violations of either property indicate an implementation, caching, feasibility, or optimization error rather than an empirical result.

In [14]:
# Evaluate all calendars under the same full displacement objective.
three_policy_run = run_three_policy_grid(
    schedule_system=primary_schedule_system,
    naive_schedule_system=naive_schedule_system,
    draws_by_product=draws_by_product,
    weekly_profiles=weekly_profiles,
    action_sets=action_sets,
    alpha_values=ALPHA_GRID, capacities=CAPACITIES,
    time_limit_seconds=MILP_TIME_LIMIT_SECONDS,
)
three_policy_results = three_policy_run["results"]
policy_results = policy_results.merge(
    three_policy_results[[
        "capacity", "alpha", "reimbursement_share",
        "value_piM", "value_piN", "value_piD",
        "delta_plan", "delta_disp", "delta_total",
        "naive_dynamic_profit", "forward_planning_naive_increment",
        "displacement_aware_increment", "sequential_decomposition_error",
        "naive_dynamic_schedule_signature",
    ]],
    on=["capacity", "alpha", "reimbursement_share"],
    how="left", validate="one_to_one",
)

# Recompute VDO directly to verify the stored arithmetic.
policy_results["recomputed_vdo"] = (
    policy_results["dynamic_profit"]
    - policy_results["myopic_profit"]
)

maximum_vdo_error = float(
    (
        policy_results["vdo"]
        - policy_results["recomputed_vdo"]
    ).abs().max()
)

if maximum_vdo_error > 1e-6:
    raise AssertionError(
        "Stored and recomputed VDO disagree. "
        f"Maximum error: {maximum_vdo_error}"
    )

if policy_results[["naive_dynamic_profit", "sequential_decomposition_error"]].isna().any().any():
    raise AssertionError("Three-policy evaluation is incomplete.")
for column in ["dynamic_profit", "myopic_profit", "vdo"]:
    if not np.allclose(policy_results[column], three_policy_results[column], atol=1e-6):
        raise AssertionError(f"Two- and three-policy {column} values disagree.")
if policy_results["sequential_decomposition_error"].abs().max() > 1e-8:
    raise AssertionError("Sequential three-policy decomposition fails.")
if not np.allclose(
    policy_results["delta_plan"] + policy_results["delta_disp"],
    policy_results["delta_total"], atol=1e-8,
):
    raise AssertionError("Delta_plan plus delta_disp must equal delta_total.")

# The dynamic feasible set includes the myopic calendar, so negative VDO beyond
# numerical tolerance indicates an optimization or implementation failure.
negative_vdo = policy_results.loc[
    policy_results["vdo"] < -1e-6
]
if not negative_vdo.empty:
    raise AssertionError(
        "Dynamic profit falls below myopic profit "
        "at one or more grid points."
    )

# Relaxing weekly capacity cannot lower the optimized dynamic objective.
dynamic_wide = (
    policy_results.pivot(
        index="alpha",
        columns="capacity",
        values="dynamic_profit",
    )
    .sort_index()
    .reindex(columns=list(CAPACITIES))
)

capacity_differences = dynamic_wide.diff(axis=1)

if (
    capacity_differences[list(CAPACITIES)[1:]] < -1e-6
).any().any():
    raise AssertionError(
        "Dynamic profit is not nondecreasing in weekly capacity."
    )

In [15]:
# Confirm that the optimized results cover the complete configured grid.
expected_policy_rows = len(ALPHA_GRID) * len(CAPACITIES)

if len(policy_results) != expected_policy_rows:
    raise RuntimeError(
        f"Expected {expected_policy_rows} policy rows, "
        f"but obtained {len(policy_results)}."
    )

actual_alphas = np.sort(
    policy_results["alpha"].astype(float).unique()
)
actual_capacities = np.sort(
    policy_results["capacity"].unique()
)
configured_capacities = np.sort(
    np.asarray(CAPACITIES)
)

if (
    len(actual_alphas) != len(ALPHA_GRID)
    or not np.allclose(actual_alphas, ALPHA_GRID)
):
    raise RuntimeError(
        "The optimized alpha values do not match ALPHA_GRID. "
        f"Expected {ALPHA_GRID}; obtained {actual_alphas}."
    )

if (
    len(actual_capacities) != len(configured_capacities)
    or not np.array_equal(actual_capacities, configured_capacities)
):
    raise RuntimeError(
        "The optimized capacities do not match CAPACITIES. "
        f"Expected {configured_capacities}; "
        f"obtained {actual_capacities}."
    )

print(
    "Validation passed:",
    "VDO arithmetic, dynamic dominance, capacity monotonicity, "
    "and complete policy-grid coverage.",
)

optimization_summary = pd.DataFrame(
    {
        "quantity": [
            "Products",
            "Funding grid points",
            "Weekly capacities",
            "Policy-result rows",
            "Selected washout",
            "Minimum VDO",
            "Maximum VDO",
        ],
        "value": [
            len(products),
            len(ALPHA_GRID),
            len(CAPACITIES),
            len(policy_results),
            SELECTED_WASHOUT,
            policy_results["vdo"].min(),
            policy_results["vdo"].max(),
        ],
    }
)

display(optimization_summary)

Validation passed: VDO arithmetic, dynamic dominance, capacity monotonicity, and complete policy-grid coverage.


,quantity,value
0,Products,8.000000
1,Funding grid points,101.000000
2,Weekly capacities,4.000000
3,Policy-result rows,404.000000
4,Selected washout,36.000000
5,Minimum VDO,0.000000
6,Maximum VDO,2502.842114


## 5. Policy-optimization artifact

The final artifact records the selected economic profile, washout horizon, schedule system, policy grid, calibrated inputs, supported actions, and the metadata required by the results and boundary-refinement notebooks.

In [16]:
# Audit the multiplier chain for the displacement-aware calendar at each grid point.
multiplier_audit_frames = [
    demand_multiplier_audit(schedule["dynamic"], draws_by_product, weekly_profiles, primary_planning)
    .assign(reimbursement_share=share, capacity=capacity)
    for (share, capacity), schedule in three_policy_run["schedules"].items()
]
multiplier_audit = pd.concat(multiplier_audit_frames, ignore_index=True)
multiplier_audit.to_csv(TABLE_DIR / "policy_demand_multiplier_audit.csv", index=False)

# Define the canonical policy artifact and reporting-table paths.
POLICY_ARTIFACT_PATH = POLICY_ARTIFACT_DIR / "policy_optimization.pkl"
POLICY_RESULTS_PATH = TABLE_DIR / "policy_results.csv"
WASHOUT_RESULTS_PATH = TABLE_DIR / "policy_washout_results.csv"
WASHOUT_STABILITY_PATH = TABLE_DIR / "policy_washout_stability.csv"
PRODUCT_DECOMPOSITION_PATH = TABLE_DIR / "policy_product_decomposition.csv"
WEEKLY_DECOMPOSITION_PATH = TABLE_DIR / "policy_weekly_decomposition.csv"
WEEKLY_PROFILE_PATH = TABLE_DIR / "weekly_economic_profiles.csv"

In [17]:
# Bundle the exact inputs, selected horizon, schedule system, policy outcomes,
# and decomposition tables required for downstream analysis and reproducibility.
artifact = {
    "selected_washout": SELECTED_WASHOUT,
    "reimbursement_grid": ALPHA_GRID,
    "alpha_grid": ALPHA_GRID,
    "capacities": CAPACITIES,
    "baseline_support": BASELINE_SUPPORT,
    "economic_profile_mode": ECONOMIC_PROFILE_MODE,
    "forecast_information_mode": policy_config.forecast_information_mode,
    "initial_conditions": policy_config.initial_conditions,
    "source_weeks": source_weeks,
    "forecast_information_audit": forecast_information_audit.to_dict(),
    "products": products,
    "action_sets": action_sets,
    "support_table": support_table,
    "weekly_profile_table": weekly_profile_table,
    "weekly_profiles": weekly_profiles,
    "draw_frame": draw_frame,
    "draws_by_product": draws_by_product,
    "schedule_system": primary_schedule_system,
    "naive_schedule_system": naive_schedule_system,
    "policy_results": policy_results.drop(columns=["recomputed_vdo"]),
    "washout_results": washout_results,
    "washout_stability": washout_stability,
    "schedules": primary_run["schedules"],
    "three_policy_schedules": three_policy_run["schedules"],
    "product_decomposition": primary_run["product_decomposition"],
    "weekly_decomposition": primary_run["weekly_decomposition"],
    "multiplier_audit": multiplier_audit,
}

In [18]:
# Save the complete binary artifact and human-readable CSV outputs.
save_pickle(artifact, POLICY_ARTIFACT_PATH)

policy_results.drop(columns=["recomputed_vdo"]).to_csv(
    POLICY_RESULTS_PATH,
    index=False,
)
washout_results.to_csv(WASHOUT_RESULTS_PATH, index=False)
washout_stability.to_csv(WASHOUT_STABILITY_PATH, index=False)
primary_run["product_decomposition"].to_csv(
    PRODUCT_DECOMPOSITION_PATH,
    index=False,
)
primary_run["weekly_decomposition"].to_csv(
    WEEKLY_DECOMPOSITION_PATH,
    index=False,
)
weekly_profile_table.to_csv(WEEKLY_PROFILE_PATH, index=False)

print("Saved:", POLICY_ARTIFACT_PATH)
print("Selected washout:", SELECTED_WASHOUT)
print("Policy rows:", len(policy_results))

# Notebook 07 consumes this exactness audit.  Generate it from the just-saved
# canonical policy artifact so the 01--08 sequence has no manual script step.
pruning_audit_script = PROJECT_ROOT / "scripts" / "verify_candidate_pruning.py"
subprocess.run(
    [
        sys.executable,
        str(pruning_audit_script),
        "--artifact",
        str(POLICY_ARTIFACT_PATH),
        "--output",
        str(TABLE_DIR / "candidate_pruning_exactness.csv"),
    ],
    cwd=PROJECT_ROOT,
    check=True,
)

Saved: C:\Users\janza\Desktop\dynamic_promotion_planning_renamed_workflow_0.3.0\artifacts\policy\empirical_bayes_price_consistent\policy_optimization.pkl
Selected washout: 36
Policy rows: 404


CompletedProcess(args=['C:\\Users\\janza\\price-of-extrapolation\\.venv\\Scripts\\python.exe', 'C:\\Users\\janza\\Desktop\\dynamic_promotion_planning_renamed_workflow_0.3.0\\scripts\\verify_candidate_pruning.py', '--artifact', 'C:\\Users\\janza\\Desktop\\dynamic_promotion_planning_renamed_workflow_0.3.0\\artifacts\\policy\\empirical_bayes_price_consistent\\policy_optimization.pkl', '--output', 'C:\\Users\\janza\\Desktop\\dynamic_promotion_planning_renamed_workflow_0.3.0\\results\\empirical_bayes_price_consistent\\tables\\candidate_pruning_exactness.csv'], returncode=0)

In [19]:
washout_report = washout_stability.copy()

washout_report["jointly_stable"] = (
    washout_report["vdo_stable"]
    & washout_report["terminal_state_small"]
)

display(
    washout_report[
        [
            "washout_horizon",
            "maximum_absolute_vdo_change",
            "maximum_terminal_state",
            "vdo_stable",
            "terminal_state_small",
            "jointly_stable",
        ]
    ]
)

print("Selection reason:", selection_reason)
print("Selected washout:", SELECTED_WASHOUT)

,washout_horizon,maximum_absolute_vdo_change,maximum_terminal_state,vdo_stable,terminal_state_small,jointly_stable
0,36,NaN,0.008492,False,True,False


Selection reason: fallback to longest configured horizon
Selected washout: 36
